#### I haven't been able to create a model that accureately predicts Fantasy Score so now my plan is to try and create a smaller model for each variable that factors into the Yahoo Fantasy Points.

# Strikeouts Model

### Checking for Correlations with new_strikeout

In [35]:
import pandas as pd
# Creating initial Data Frame
df = pd.read_csv("Clean_2015_Pitching_Data.csv", sep = ",")

# Eliminating String Features to be able to run Correlation Analysis
# Eliminating all target features except new_strikeout
new_strikeout_df = df.drop(['NAME', 'ID', 'new_yahoo', 'new_IP', 'new_win', 'new_loss', 'new_hit', 'new_earned_run', 'new_walk'], axis=1)

In [36]:
import pandas as pd

# Calculate the correlation matrix
correlation_matrix = new_strikeout_df.corr()

# Select the target variable column
target_variable = "new_strikeout" 

# Get correlations with the target variable
target_correlations = correlation_matrix[target_variable].sort_values(ascending=False)

# Print the correlations (optional)
print(target_correlations)

new_strikeout         1.000000
k_percent             0.640059
strikeout             0.597983
whiff_percent         0.554604
p_swinging_strike     0.544201
                        ...   
xslg                 -0.453699
xobp                 -0.469947
xwoba                -0.496707
iz_contact_percent   -0.525986
xba                  -0.544418
Name: new_strikeout, Length: 97, dtype: float64


### Model 1 - Single Feature Model

In [37]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(new_strikeout_df,
test_size=0.2, random_state=123)
print(len(train_set), len(test_set))

from sklearn.linear_model import LinearRegression
reg = LinearRegression()

X = train_set[['xba']]
y = train_set['new_strikeout']
reg.fit(X, y)

print("The bias is " , reg.intercept_)
print("The feature coefficients are ", reg.coef_)
print("The score for the training set is", reg.score(X,y))

# Check the performance on the test set
X_test = test_set[['xba']]
y_test = test_set['new_strikeout']
print("The score for the test set is", reg.score(X_test,y_test))

332 83
The bias is  428.95174012019083
The feature coefficients are  [-1110.08929402]
The score for the training set is 0.3000074886108862
The score for the test set is 0.2748983537709372


#### Simple Linear Model Performance
| Feature   | Training   | Test    |
| -----     | -----      | -----   |
| k_percent | 0.42 | 0.36 |
| strikeout | 0.38 | 0.25 |
| whiff_percent | 0.30 | 0.32 |
| p_swinging_strike | 0.31 | 0.25 |
| xba | 0.30 | 0.27 |




### Model 2 - Multi-Feature Model

In [43]:
X = train_set[['k_percent','strikeout', 'whiff_percent', 'xba', 'p_swinging_strike']]
y = train_set['new_strikeout']
reg.fit(X, y)

print("The bias is " , reg.intercept_)
print("The feature coefficients are ", reg.coef_)
print("The score for the training set is", reg.score(X,y))

# Check the performance on the test set
X_test = test_set[['k_percent','strikeout', 'whiff_percent', 'xba', 'p_swinging_strike']]
y_test = test_set['new_strikeout']
print("The score for the test set is", reg.score(X_test,y_test))

The bias is  27.351810928793185
The feature coefficients are  [ 5.50801027  0.21505068 -1.71761361 10.48325417  0.04754484]
The score for the training set is 0.4496081950641221
The score for the test set is 0.34155856243446836


#### Multi-Feature Model Performance
| # Features | Features | Training   | Test    |
| -----     | -----      | -----   | ----- |
| 2 | strikeout, k_percent | 0.45 | 0.35 |
| 2 | k_percent, whiff_percent | 0.42 | 0.35 |
| 2 | k_percent, p_swinging_strike | 0.43 | 0.36 |
| 2 | k_percent, xba | 0.42 | 0.36 |
| 2 | strikeout, whiff_percent | 0.41 | 0.33 |
| 2 | whiff_percent, xba | 0.35 | 0.35 |
| 3 | k_percent, strikeout, xba | 0.45 | 0.35 |
| 4 | k_percent, strikeout, xba, whiff_percent | 0.45 | 0.34 |
| 5 | k_percent, strikeout, xba, whiff_percent, p_swinging_strike | 0.45 | 0.34 |







### Model 3 - Decision Tree Model

In [50]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

# Separate features (X) and target variable (y)
X = new_strikeout_df.drop(['new_strikeout'], axis =1)
y = new_strikeout_df['new_strikeout']

# Split data into training and testing sets (test_size=0.2 for 20% test data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [65]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import precision_score, recall_score

# Create the decision tree regression model
model = DecisionTreeRegressor(max_depth=3)

# Train the model on the training data
model.fit(X_train, y_train)

# Make predictions on testing data
y_pred = model.predict(X_test)

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

Mean Squared Error: 2138.0212190110615
R-squared: 0.12592914194383165


#### Decision Tree Results
| Depth | MSE   | R2    |
| ----- | ----- | ----- |
| 5     | 2632 | -.07?? |
| 4     | 1946 | .20 |
| 3     | 2138 | .13 |
| 2     | 1591 | .35 |
| 1     | 1783 | .27 |


In [66]:
# Access feature importances
feature_importances = model.feature_importances_

# Sort features and importances together by importance (descending order)
sorted_features_and_importances = sorted(zip(X.columns, feature_importances), key=lambda x: x[1], reverse=True)

# Print features in order of importance
for feature, importance in sorted_features_and_importances:
  print(f"{feature}: {importance:.4f}")

k_percent: 0.7880
p_lob: 0.0684
z_swing_percent: 0.0679
strikeout: 0.0527
meatball_percent: 0.0229
last_year: 0.0000
player_age: 0.0000
last_year_Yahoo: 0.0000
IP: 0.0000
pa: 0.0000
ab: 0.0000
hit: 0.0000
single: 0.0000
double: 0.0000
triple: 0.0000
home_run: 0.0000
walk: 0.0000
bb_percent: 0.0000
batting_avg: 0.0000
slg_percent: 0.0000
on_base_percent: 0.0000
on_base_plus_slg: 0.0000
isolated_power: 0.0000
babip: 0.0000
p_earned_run: 0.0000
p_run: 0.0000
p_win: 0.0000
p_loss: 0.0000
p_ball: 0.0000
p_called_strike: 0.0000
p_swinging_strike: 0.0000
p_total_ball: 0.0000
p_total_bases: 0.0000
p_total_strike: 0.0000
p_total_swinging_strike: 0.0000
xba: 0.0000
xslg: 0.0000
woba: 0.0000
xwoba: 0.0000
xobp: 0.0000
xiso: 0.0000
xbadiff: 0.0000
xslgdiff: 0.0000
wobadiff: 0.0000
exit_velocity_avg: 0.0000
launch_angle_avg: 0.0000
sweet_spot_percent: 0.0000
barrel_batted_rate: 0.0000
solidcontact_percent: 0.0000
poorlyunder_percent: 0.0000
poorlytopped_percent: 0.0000
poorlyweak_percent: 0.0000
ha

### Model 4 - LASSO Model

In [89]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso

# Create the LASSO model with alpha (regularization parameter)
model = Lasso(alpha=11)  # Adjust alpha as needed

# Train the model on the training data
model.fit(X_train, y_train)

# Make predictions on testing data
y_pred = model.predict(X_test)

# Evaluate model performance (e.g., using mean squared error)
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

Mean Squared Error: 1254.5795940723249
R-squared: 0.48709982270532237


c:\Users\wadej\Documents\Capstone\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.441e+03, tolerance: 6.890e+01
  model = cd_fast.enet_coordinate_descent(


#### Lasso Alpha Analysis
| Alpha | MSE   | R2    |
| ----- | ----- | ----- |
| .001  | 1398 | .4281 |
| .01   | 1401 | .4217 |
| .1    | 1372 | .4388 |
| 1     | 1354 | .4462 |
| 10    | 1255 | .4865 |
| 11    | 1254 | .4871 |

In [91]:
# Access coefficients
lasso_coefficients = model.coef_

# Sort features and coefficients together by absolute coefficient value (descending order)
sorted_features_and_coefficients = sorted(zip(X.columns, lasso_coefficients), key=lambda x: abs(x[1]), reverse=True)

# Print features and coefficients in sorted order
for feature_name, coefficient in sorted_features_and_coefficients:
  print(f"{feature_name}: {coefficient:.4f}")

strikeout: 0.9183
player_age: -0.8347
p_lob: 0.2969
double: 0.2483
pa: -0.1203
p_total_strike: -0.1110
p_ball: 0.1061
p_total_ball: -0.1030
p_total_bases: -0.0764
fastball_avg_spin: 0.0652
in_zone: 0.0499
last_year_Yahoo: -0.0483
out_zone_swing: 0.0468
total_pitches: 0.0400
p_swinging_strike: -0.0256
out_zone: -0.0254
p_total_swinging_strike: 0.0122
offspeed_avg_spin: -0.0096
p_called_strike: -0.0096
pitch_count_breaking: 0.0056
pitch_count_fastball: 0.0047
edge: 0.0014
breaking_avg_spin: 0.0013
ab: -0.0013
pitch_count_offspeed: 0.0002
last_year: -0.0000
IP: -0.0000
hit: 0.0000
single: -0.0000
triple: 0.0000
home_run: -0.0000
walk: 0.0000
k_percent: 0.0000
bb_percent: 0.0000
batting_avg: 0.0000
slg_percent: -0.0000
on_base_percent: 0.0000
on_base_plus_slg: -0.0000
isolated_power: -0.0000
babip: 0.0000
p_earned_run: 0.0000
p_run: 0.0000
p_win: 0.0000
p_loss: 0.0000
xba: -0.0000
xslg: -0.0000
woba: -0.0000
xwoba: -0.0000
xobp: 0.0000
xiso: -0.0000
xbadiff: 0.0000
xslgdiff: 0.0000
wobadif

# Final Results

| Method | Specifics   | R2   |
| ----- | ----- | ----- |
| LASSO | Alpha = 11 | 0.4871 |
| Multi-Feature Regression | Features = strikeout, k_percent | 0.45 / 0.35 |
| Simple Linear Regression | Feature = k_percent | 0.42 / 0.36 |
| Decision Tree  | Depth = 2 | .35 |
| Simple Linear Regression | Feature = strikeout | 0.32 / 0.26 |
